# 가중치 초기화 전략
## Xavier 초기화 / He 초기화 / GPT-2 초기화

- Tutorial ID: `adv-6-1`
- Section ID: `adv-6-1-1`

---

### 📌 이 노트북에서 다루는 것

딥러닝 모델을 학습시키기 전, 각 뉴런의 **가중치(weight)에 적절한 초기값**을 설정하는 것은
학습 성공 여부를 결정하는 핵심 요소입니다.

**왜 초기화가 중요한가요?**

신호(텐서 값)가 수십~수백 개의 레이어를 통과할 때:
- 가중치가 너무 크면 → 값이 폭발적으로 증가 🔴 **(Gradient Exploding)**
- 가중치가 너무 작으면 → 값이 0으로 사라짐 🔵 **(Gradient Vanishing)**
- 가중치가 딱 맞으면 → 값이 안정적으로 유지 🟢 **(이것이 목표!)**

---

### 📋 실험 순서

| # | 주제 | 핵심 포인트 |
|---|------|-------------|
| 1 | 분산(Variance) 이해 | 초기화와 분산의 관계 직관 |
| 2 | 나쁜 초기화: 0으로 초기화 | 대칭성 문제(Symmetry Problem) |
| 3 | 나쁜 초기화: 스케일 없는 랜덤 | 분산 폭발 / 소실 실험 |
| 4 | Xavier 초기화 | tanh / sigmoid 활성화 함수와 최적 궁합 |
| 5 | He 초기화 | ReLU 활성화 함수와 최적 궁합 |
| 6 | GPT-2 초기화 | 잔차 연결(Residual) 누적 분산 보정 |
| 7 | 종합 비교 | 모든 방법을 한 화면에서 비교 |

---

> 💡 **코드 읽는 법**: 숫자 하나하나보다 **"분산이 레이어마다 어떻게 변하는가"** 에 집중하세요.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
import platform
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────
# 한글 폰트 설정
#
# [Google Colab 사용자]
#   아래 두 줄의 '#'을 지우고 셀을 실행한 뒤,
#   메뉴 → 런타임 → 런타임 다시 시작 을 클릭하세요.
#
# !apt-get install -y fonts-nanum > /dev/null 2>&1
# !fc-cache -fv > /dev/null 2>&1
#
#   그 다음 font_family = 'NanumGothic' 으로 바꾸세요.
# ─────────────────────────────────────────────────────────────────

if platform.system() == 'Darwin':       # macOS
    font_family = 'AppleGothic'
elif platform.system() == 'Windows':    # Windows
    font_family = 'Malgun Gothic'
else:                                    # Linux / Google Colab
    font_family = 'DejaVu Sans'         # 기본값 (한글 깨질 수 있음)
    # font_family = 'NanumGothic'       # 나눔고딕 설치 후 이 줄로 교체

matplotlib.rc('font', family=font_family)
matplotlib.rc('axes', unicode_minus=False)  # 마이너스 기호(-) 깨짐 방지
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'

np.random.seed(42)  # 재현성 시드 고정

print("=" * 60)
print("  가중치 초기화 전략 실습")
print("=" * 60)
print(f"  NumPy 버전 : {np.__version__}")
print(f"  폰트 설정  : {font_family}")
print(f"  난수 시드  : 42 (재현성 보장)")
print()
print("준비 완료! 아래 셀들을 순서대로 실행하세요.")


---

## 섹션 1: 분산(Variance) 이해하기

초기화를 배우기 전에, **"분산이 무엇인지"** 먼저 직관적으로 이해해봅니다.

### 분산(Variance)이란?

분산은 **숫자들이 평균 주변에 얼마나 퍼져 있는지**를 나타내는 척도입니다.

```
예시:
  [1, 1, 1, 1]       → 모두 같음   → 분산 = 0
  [-1, 0, 0, 1]      → 약간 퍼짐  → 분산 = 0.5
  [-100, 0, 0, 100]  → 많이 퍼짐  → 분산 = 5000
```

### 왜 딥러닝에서 분산이 중요한가요?

레이어가 쌓일수록 값이 전파되는데:
- **분산 > 1** → 레이어마다 값이 커져서 결국 무한대로 폭발 🔴
- **분산 < 1** → 레이어마다 값이 작아져서 결국 0으로 소실 🔵
- **분산 ≈ 1** → 안정적으로 전파 ✅

> **목표**: 레이어를 아무리 많이 통과해도 분산이 ≈ 1로 유지되는 초기화!


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 분산이 서로 다른 세 가지 신호를 만들어 비교합니다.
#
# np.random.randn(d)
#   → 평균 0, 표준편차(std) 1인 정규분포에서 d개 숫자를 뽑습니다.
#
# 여기에 상수를 곱하면 표준편차가 스케일됩니다:
#   * 0.001  →  std ≈ 0.001,  분산 ≈ 0.000001  (소실 위험)
#   * 1.0    →  std ≈ 1.0,    분산 ≈ 1.0        (이게 목표!)
#   * 10.0   →  std ≈ 10.0,   분산 ≈ 100.0      (폭발 위험)
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("[섹션 1] 분산(Variance) 직관 이해")
print("=" * 60)

np.random.seed(42)
d = 512  # 512차원 벡터 (실제 딥러닝 레이어 크기)

x_small  = np.random.randn(d) * 0.001   # 매우 작은 분산 → 소실 위험
x_normal = np.random.randn(d) * 1.0     # 적절한 분산   → 목표!
x_large  = np.random.randn(d) * 10.0    # 매우 큰 분산  → 폭발 위험

print()
print("처음 5개 값 샘플 비교:")
print(f"  🔵 작은 분산 (std=0.001) : {np.round(x_small[:5],  6)}")
print(f"  🟢 적절한 분산 (std=1.0)  : {np.round(x_normal[:5], 4)}")
print(f"  🔴 큰 분산 (std=10.0)    : {np.round(x_large[:5],  2)}")

print()
print("분산 값 비교:")
print(f"  🔵 작은 분산  : {np.var(x_small):.8f}  ← 거의 0  (소실)")
print(f"  🟢 적절한 분산: {np.var(x_normal):.4f}      ← 목표! ✅")
print(f"  🔴 큰 분산    : {np.var(x_large):.4f}   ← 너무 큼 (폭발)")

print()
print("─" * 60)
print("💡 딥러닝에서는 레이어를 아무리 많이 쌓아도")
print("   분산이 1.0 근처를 유지해야 학습이 안정됩니다.")


In [ ]:
# 분산 차이를 히스토그램으로 시각화합니다.
# 히스토그램: x축 = 값의 크기, y축 = 해당 값이 몇 번 등장했는지
# → 좁고 뾰족 = 분산 작음 / 넓게 퍼짐 = 분산 큼

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Distribution of Values with Different Variances\n(분산이 다른 세 신호의 분포)',
             fontsize=13, fontweight='bold', y=1.02)

plot_cfg = [
    (x_small,  'steelblue',   '🔵 Small Var (소실 위험)\nstd=0.001',  (-0.005, 0.005)),
    (x_normal, 'forestgreen', '🟢 Healthy Var (목표!)\nstd=1.0',       (-4,     4)),
    (x_large,  'crimson',     '🔴 Large Var (폭발 위험)\nstd=10.0',    (-45,   45)),
]

for ax, (data, color, title, xlim) in zip(axes, plot_cfg):
    ax.hist(data, bins=60, color=color, alpha=0.75, edgecolor='white')
    ax.set_title(title, fontsize=11, pad=10)
    ax.set_xlabel('Value (값)')
    ax.set_ylabel('Count (빈도)')
    ax.set_xlim(xlim)
    ax.axvline(0, color='black', linestyle='--', alpha=0.4, linewidth=1.2)
    ax.text(0.97, 0.95, f'Var={np.var(data):.6f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("→ 가운데 그림(분산≈1.0)처럼 적당히 퍼진 분포가 이상적입니다.")
print("→ 왼쪽(너무 좁음)과 오른쪽(너무 넓음)은 깊은 네트워크에서 문제가 됩니다.")


---

## 섹션 2: 나쁜 초기화 ① — 모두 0으로 초기화

### 아이디어: "일단 0으로 시작하면 안전하지 않을까?"

직관적으로는 맞는 것 같지만, **절대로 하면 안 됩니다!**

### 왜 나쁜가요? — 대칭성 문제 (Symmetry Problem)

모든 가중치가 0이면:

```
입력 x = [1, 2, 3]
가중치 W = [[0, 0, 0],   ← 모두 0
            [0, 0, 0]]

출력 = W @ x = [0, 0]   ← 모두 0!
```

**더 큰 문제: 역전파(Backpropagation)**

```
모든 뉴런 가중치 동일
   → 모든 뉴런이 동일한 그래디언트를 받음
   → 모든 뉴런이 똑같이 업데이트됨
   → 학습이 끝나도 모든 뉴런이 동일한 상태
   → 수천 개의 뉴런이 있어도 실질적으로 뉴런 1개와 같음!
```

> 이것을 **"대칭성 파괴 실패(Failure to Break Symmetry)"** 라고 합니다.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 0 초기화의 문제를 실제 숫자로 확인합니다.
#
# 핵심:
#   - 모든 가중치가 0 → 입력이 무엇이든 출력은 항상 0
#   - 역전파 시 모든 뉴런이 동일한 그래디언트 → 영원히 동일 상태 유지
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("[섹션 2] 나쁜 초기화 ① 모두 0으로 초기화")
print("=" * 60)

n_input  = 4   # 입력 뉴런 수
n_hidden = 3   # 히든 뉴런 수

W_zero = np.zeros((n_hidden, n_input))       # 0으로 초기화
x_input = np.array([1.0, -2.0, 3.0, -0.5])  # 임의의 입력 벡터

print()
print(f"입력 벡터 x: {x_input}")
print()
print(f"0으로 초기화된 가중치 행렬 W (shape {W_zero.shape}):")
print(W_zero)

output_zero = W_zero @ x_input
print()
print(f"레이어 출력 W @ x = {output_zero}")
print("→ 어떤 입력을 넣어도 출력은 항상 [0, 0, 0]!")

print()
print("─" * 60)
print("📌 뉴런별로 따져보면:")
for i in range(n_hidden):
    dot = np.dot(W_zero[i], x_input)
    print(f"   뉴런 #{i+1}: {W_zero[i]} · {x_input} = {dot}")
print()
print("→ 세 뉴런 모두 완전히 동일한 계산 → 동일한 그래디언트")
print("→ 학습을 아무리 해도 뉴런들이 서로 달라지지 않음!")

print()
print("─" * 60)
print("✅ 올바른 랜덤 초기화와 비교:")
np.random.seed(42)
W_rand = np.random.randn(n_hidden, n_input) * 0.1
output_rand = W_rand @ x_input
print()
print(f"랜덤 초기화 W:")
print(np.round(W_rand, 4))
print()
print(f"랜덤 초기화 출력: {np.round(output_rand, 4)}")
print("→ 각 뉴런이 서로 다른 출력 → 다른 그래디언트 → 서로 다른 특징 학습!")

print()
print("💡 결론: 가중치를 0으로 초기화하면 절대 안 됩니다.")
print("   반드시 '서로 다른 값(랜덤)'으로 초기화해야 합니다.")


---

## 섹션 3: 나쁜 초기화 ② — 스케일 없는 랜덤 초기화

### 아이디어: "랜덤으로 초기화하면 되지 않을까?"

랜덤 초기화는 대칭성 문제를 해결하지만, **크기(스케일)를 고려하지 않으면** 다른 문제가 생깁니다.

### 수학적으로 이해하기

레이어 출력 `y = W @ x` 에서 분산이 어떻게 변하는지 계산해봅시다.

```
d개의 입력이 있을 때 (d = 입력 차원):
  y_j = w_1j * x_1  +  w_2j * x_2  +  ...  +  w_dj * x_d

통계 법칙(독립 확률변수의 분산 합):
  Var(y) ≈ d × Var(w) × Var(x)
           ↑
           d배 증폭됨!
```

**예시**: `d = 512`, `Var(w) = 1.0`, `Var(x) = 1.0`
- `Var(y) ≈ 512` — 한 레이어 통과만으로 512배 증폭!
- 50개 레이어 후: `512^50` — 우주의 원자 수보다 큰 숫자 💥

반대로 `std(w) = 0.01` 이면:
- `Var(y) ≈ 512 × 0.0001 = 0.0512` — 매 레이어마다 감소
- 50개 레이어 후: 사실상 0 💧

> **핵심**: 가중치의 크기를 차원 `d`에 맞게 조절해야 합니다!


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 스케일 없는 초기화의 두 가지 실패 케이스를 실험합니다.
#
# [변수 설명]
#   d        : 레이어 차원 (뉴런 수) = 512
#   n_layers : 레이어 깊이 = 50
#   x        : 현재 신호 벡터 (shape: [d,])
#   W        : 가중치 행렬 (shape: [d, d])
#
# [핵심 수식]
#   한 레이어 통과 후 분산: Var(W @ x) ≈ d × Var(w) × Var(x)
#   std=1.0  → d=512배 증폭  → 폭발
#   std=0.01 → 0.0512배 감소 → 소실
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("[섹션 3] 나쁜 초기화 ② 스케일 없는 랜덤 초기화")
print("=" * 60)

d = 512
n_layers = 50

# ─── 케이스 1: 분산 폭발 (std=1.0) ────────────────────────────
print()
print("📌 케이스 1: W ~ N(0, 1.0) → 분산 폭발")
print(f"   이론: 한 레이어마다 분산이 ~{d}배 증폭됩니다")
print()

np.random.seed(42)
x = np.random.randn(d)         # 초기 신호 (분산 ≈ 1.0)
variances_explode = []

for i in range(n_layers):
    W = np.random.randn(d, d)  # std=1.0, 스케일링 없음
    x = W @ x                  # 레이어 통과
    v = np.var(x)
    variances_explode.append(v)

    if i < 3 or i == n_layers // 2 or i >= n_layers - 2:
        if np.isnan(v) or np.isinf(v):
            print(f"  레이어 {i+1:3d}: 수치 오버플로우 ∞ 🔴")
        elif v > 1e10:
            print(f"  레이어 {i+1:3d}: 분산 = {v:.2e}  🔴 폭발!")
        else:
            print(f"  레이어 {i+1:3d}: 분산 = {v:.4f}")

# ─── 케이스 2: 분산 소실 (std=0.01) ───────────────────────────
print()
print("─" * 60)
print()
print("📌 케이스 2: W ~ N(0, 0.01) → 분산 소실")
print(f"   이론: 한 레이어마다 분산이 ~{d * 0.01**2:.4f}배로 감소합니다")
print()

np.random.seed(42)
x = np.random.randn(d)
variances_vanish = []

for i in range(n_layers):
    W = np.random.randn(d, d) * 0.01   # 너무 작은 std
    x = W @ x
    v = np.var(x)
    variances_vanish.append(v)

    if i < 3 or i == n_layers // 2 or i >= n_layers - 2:
        print(f"  레이어 {i+1:3d}: 분산 = {v:.2e}")

print()
print("─" * 60)
print()
print("💡 결론:")
print("  std=1.0  → 분산이 매 레이어마다 ~512배 증폭 → 수치 폭발 🔴")
print("  std=0.01 → 분산이 매 레이어마다 급격히 감소 → 수치 소실 🔵")
print("  → 분산이 안정적으로 유지되도록 스케일을 d에 맞게 조절해야 합니다!")


In [ ]:
# 분산 폭발과 소실을 시각화합니다.
# semilogy: y축을 로그 스케일로 표시 (10, 100, 1000, ... 처럼 10배씩 증가)
# → 엄청나게 큰 값과 엄청나게 작은 값을 한 화면에서 비교 가능

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bad Initialization: Variance Explosion & Vanishing\n(나쁜 초기화: 분산 폭발과 소실)',
             fontsize=13, fontweight='bold', y=1.02)

layer_idx = list(range(1, n_layers + 1))

# 왼쪽: 분산 폭발 (std=1.0)
ax = axes[0]
safe_explode = [min(v, 1e50) if not (np.isnan(v) or np.isinf(v)) else 1e50
                for v in variances_explode]
ax.semilogy(layer_idx, safe_explode, 'r-o', markersize=4, linewidth=2, label='std=1.0 (폭발)')
ax.axhline(y=1.0, color='green', linestyle='--', linewidth=2, alpha=0.8, label='목표 분산=1.0')
ax.set_title('케이스 1: 분산 폭발 (std=1.0)\n→ 레이어마다 ~512배씩 증가', fontsize=11)
ax.set_xlabel('레이어 번호 (Layer Number)')
ax.set_ylabel('분산 (Variance, log scale)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#fff5f5')

# 오른쪽: 분산 소실 (std=0.01)
ax = axes[1]
safe_vanish = [max(v, 1e-300) for v in variances_vanish]
ax.semilogy(layer_idx, safe_vanish, 'b-o', markersize=4, linewidth=2, label='std=0.01 (소실)')
ax.axhline(y=1.0, color='green', linestyle='--', linewidth=2, alpha=0.8, label='목표 분산=1.0')
ax.set_title('케이스 2: 분산 소실 (std=0.01)\n→ 레이어마다 급격히 감소', fontsize=11)
ax.set_xlabel('레이어 번호 (Layer Number)')
ax.set_ylabel('분산 (Variance, log scale)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f5f5ff')

plt.tight_layout()
plt.show()

print()
print("→ 두 그래프 모두 초록 점선(목표=1.0)에서 크게 벗어납니다.")
print("→ 이상적인 초기화는 모든 레이어에서 초록 점선 근처를 유지해야 합니다!")


---

## 섹션 4: Xavier 초기화 (Glorot 초기화)

> Glorot & Bengio (2010), *"Understanding the difficulty of training deep feedforward neural networks"*

### 핵심 아이디어

레이어를 통과해도 분산이 유지되도록, **가중치 크기를 차원 `d`에 반비례**하게 설정합니다.

### 수학적 유도 (단계별)

```
1단계: 출력 분산 계산
   y = w_1*x_1 + w_2*x_2 + ... + w_d*x_d  (d개 입력의 선형 결합)

   Var(y) = d × Var(w) × Var(x)        (독립 확률변수의 분산 합 법칙)

2단계: 분산 유지 조건 설정
   목표: Var(y) = Var(x)  →  입출력 분산이 같아야 한다!

3단계: 역산
   d × Var(w) = 1
   Var(w) = 1/d
   std(w) = 1/√d

4단계: Xavier (역전파도 고려)
   입력(fan_in)과 출력(fan_out)을 모두 고려:
   std = √(2 / (fan_in + fan_out))
```

**용어 설명:**
- `fan_in`  = 이 레이어로 **들어오는** 연결 수 (입력 차원)
- `fan_out` = 이 레이어에서 **나가는** 연결 수 (출력 차원)

### 언제 사용하나요?

✅ `tanh`, `sigmoid` 같은 선형-유사 활성화 함수와 함께 사용할 때  
❌ ReLU와는 잘 맞지 않음 (다음 섹션에서 그 이유를 설명합니다)


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Xavier 초기화 핵심:
#   std = sqrt(2 / (fan_in + fan_out))
#
# 직관: 차원이 클수록 std를 더 작게 → 분산 증폭을 상쇄
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("[섹션 4] Xavier 초기화")
print("=" * 60)

print()
print("📐 공식: std = √(2 / (fan_in + fan_out))")
print()
print("차원별 Xavier std 값 확인:")
print()
print(f"  {'차원 (d)':>10} | {'Xavier std':>12} | {'단순 std=1.0':>14} | 차이 배율")
print("  " + "─" * 55)
for d_ex in [64, 128, 256, 512, 1024]:
    xavier_std = np.sqrt(2.0 / (d_ex + d_ex))  # fan_in=fan_out=d (정방 행렬)
    print(f"  {d_ex:>10} | {xavier_std:>12.6f} | {'1.0':>14} | 1/{1/xavier_std:.1f}")

print()
print("→ 차원이 클수록 Xavier std가 작아집니다.")
print("   d가 d배 증폭시키는 것을 std를 1/√d로 줄여 상쇄!")

print()
print("─" * 60)
print()
print("📌 Xavier 초기화 실험 (50개 레이어):")
print()

d = 512
n_layers = 50
np.random.seed(42)

x = np.random.randn(d)
print(f"  초기 분산: {np.var(x):.4f}")
variances_xavier = [np.var(x)]

for i in range(n_layers):
    fan_in  = d          # 정방 행렬: 입력 차원
    fan_out = d          # 정방 행렬: 출력 차원

    # ─── Xavier 핵심 공식 ───
    std = np.sqrt(2.0 / (fan_in + fan_out))   # ≈ 1/√d

    W = np.random.randn(d, d) * std            # 스케일된 가중치
    x = W @ x                                   # 레이어 통과
    v = np.var(x)
    variances_xavier.append(v)

    if i < 3 or i == n_layers // 2 or i >= n_layers - 2:
        ok = '✓' if 0.3 < v < 3.0 else '!'
        print(f"  레이어 {i+1:3d}: 분산 = {v:.6f}  {ok}")

print()
final_var = variances_xavier[-1]
print(f"최종 분산: {final_var:.6f}")
print(f"초기 대비 변화: {final_var:.2f}배")
print()
print(f"🎉 {n_layers}개 레이어를 통과해도 분산이 안정적으로 유지됩니다!")


In [ ]:
# Xavier 초기화를 이전 나쁜 초기화와 비교 시각화합니다.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Xavier Initialization vs Bad Initializations\n(Xavier 초기화 vs 나쁜 초기화 비교)',
             fontsize=13, fontweight='bold', y=1.02)

layer_idx_full = list(range(n_layers + 1))   # 0~50 (초기값 포함)
layer_idx_bad  = list(range(1, n_layers + 1))  # 1~50

# 왼쪽: 로그 스케일 전체 비교
ax = axes[0]
ax.semilogy(layer_idx_bad,
            [min(v, 1e50) if not (np.isnan(v) or np.isinf(v)) else 1e50
             for v in variances_explode],
            'r--', linewidth=2, alpha=0.7, label='std=1.0 (폭발)')
ax.semilogy(layer_idx_bad,
            [max(v, 1e-300) for v in variances_vanish],
            'b--', linewidth=2, alpha=0.7, label='std=0.01 (소실)')
ax.semilogy(layer_idx_full, variances_xavier,
            'g-o', linewidth=2.5, markersize=5, label='Xavier ✓')
ax.axhline(y=1.0, color='black', linestyle=':', linewidth=2, alpha=0.6, label='목표=1.0')
ax.set_title('전체 비교 (로그 스케일)\n→ Xavier만 목표 근처 유지', fontsize=11)
ax.set_xlabel('레이어 번호')
ax.set_ylabel('분산 (log scale)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 오른쪽: Xavier만 일반 스케일 확대
ax = axes[1]
ax.plot(layer_idx_full, variances_xavier, 'g-o', linewidth=2.5, markersize=5, label='Xavier')
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='목표=1.0')
ax.fill_between(layer_idx_full, 0.5, 2.0, alpha=0.12, color='green', label='허용 범위 [0.5, 2.0]')
ax.set_title('Xavier 상세 보기\n→ 50개 레이어 내내 1.0 근처 유지', fontsize=11)
ax.set_xlabel('레이어 번호')
ax.set_ylabel('분산')
ax.set_ylim(0, 3)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f8fff8')

plt.tight_layout()
plt.show()

print()
print("→ Xavier(초록)는 50개 레이어 내내 분산이 안정적입니다.")
print("→ 나쁜 초기화(빨강/파랑)와 극명한 대비를 보입니다.")


---

## 섹션 5: He 초기화 (Kaiming 초기화)

> He et al. (2015), *"Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet"*

### 왜 Xavier만으로는 부족한가요?

Xavier는 **선형(linear)** 또는 **tanh/sigmoid** 같은 활성화 함수를 가정합니다.  
하지만 현대 딥러닝의 핵심 활성화 함수는 **ReLU**입니다!

### ReLU의 특성

```python
ReLU(x) = max(0, x)

  x < 0  →  0   (음수를 모두 0으로 만듦!)
  x >= 0 →  x   (양수는 그대로)

→ 평균적으로 절반의 입력이 0이 됨
→ 유효 분산이 약 절반으로 감소
```

### Xavier + ReLU의 문제

```
Xavier 목표: Var(출력) = Var(입력)
ReLU 효과:   Var(출력) ≈ Var(입력) / 2     (절반이 0으로 잘림)
결과:         레이어마다 분산이 0.5배씩 감소 → 소실!
```

### He 초기화의 해결책

Xavier의 **√2배**를 사용해서 ReLU의 분산 감소를 보상합니다:

```
Xavier: std = √(1 / fan_in)
He:     std = √(2 / fan_in)    ← √2 ≈ 1.41배 더 큼!
           ↑
        ReLU가 분산을 절반으로 줄이는 것을 2배로 보상
```


In [ ]:
# ─────────────────────────────────────────────────────────────────
# He 초기화와 Xavier 초기화를 ReLU 환경에서 비교합니다.
#
# ReLU(x) = max(0, x)
#   - 음수 → 0, 양수 → 그대로
#   - 결과적으로 값의 절반이 0이 되어 분산이 감소
#
# He: std = sqrt(2/fan_in) = Xavier * sqrt(2)
#   - 이 sqrt(2) 인수가 ReLU의 분산 감소를 정확히 보상
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("[섹션 5] He 초기화 — ReLU 네트워크용")
print("=" * 60)

def relu(x):
    # ReLU: 음수는 0으로, 양수는 그대로
    # 딥러닝에서 가장 많이 사용되는 활성화 함수
    return np.maximum(0, x)

d = 512
n_layers = 50

print()
print("📐 공식 비교:")
d_ex = 512
print(f"   Xavier std = √(2 / (fan_in + fan_out)) = √(2/{d_ex*2}) = {np.sqrt(2.0/(d_ex*2)):.6f}")
print(f"   He std     = √(2 / fan_in)              = √(2/{d_ex})   = {np.sqrt(2.0/d_ex):.6f}")
print(f"   비율: He / Xavier = {np.sqrt(2.0/d_ex) / np.sqrt(2.0/(d_ex*2)):.4f} (≈ √2 = {np.sqrt(2):.4f})")

# ─── He + ReLU ────────────────────────────────────────────────
print()
print("─" * 60)
print()
print("📌 He 초기화 + ReLU:")
np.random.seed(42)
x = np.random.randn(d)
print(f"  초기 분산: {np.var(x):.4f}")
variances_he = [np.var(x)]

for i in range(n_layers):
    fan_in = d
    std = np.sqrt(2.0 / fan_in)   # ← He 핵심 공식

    W = np.random.randn(d, d) * std
    x = W @ x
    x = relu(x)                   # ← ReLU 적용!
    v = np.var(x)
    variances_he.append(v)

    if i < 3 or i == n_layers // 2 or i >= n_layers - 2:
        ok = '✓' if 0.1 < v < 5.0 else '!'
        print(f"  레이어 {i+1:3d}: 분산 = {v:.4f}  {ok}")

# ─── Xavier + ReLU (비교) ────────────────────────────────────
print()
print("─" * 60)
print()
print("📌 Xavier 초기화 + ReLU (비교용):")
np.random.seed(42)
x = np.random.randn(d)
print(f"  초기 분산: {np.var(x):.4f}")
variances_xavier_relu = [np.var(x)]

for i in range(n_layers):
    fan_in = fan_out = d
    std = np.sqrt(2.0 / (fan_in + fan_out))   # Xavier (He보다 작음)

    W = np.random.randn(d, d) * std
    x = W @ x
    x = relu(x)                               # ReLU 적용 (Xavier와 같은 조건)
    v = np.var(x)
    variances_xavier_relu.append(v)

    if i < 3 or i == n_layers // 2 or i >= n_layers - 2:
        print(f"  레이어 {i+1:3d}: 분산 = {v:.8e}")

print()
print("─" * 60)
print()
print("💡 결론:")
print("  He + ReLU      → 분산 안정 ✅  (ReLU 네트워크의 표준)")
print("  Xavier + ReLU  → 분산 소실 ❌  (ReLU와 궁합이 안 맞음)")


In [ ]:
# He vs Xavier + ReLU 시각화

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('He vs Xavier Initialization with ReLU Activation\n(ReLU 사용 시 He vs Xavier 비교)',
             fontsize=13, fontweight='bold', y=1.02)

layer_idx = list(range(n_layers + 1))

# 왼쪽: 일반 스케일 (차이 직관적으로 보기)
ax = axes[0]
ax.plot(layer_idx, variances_he, 'g-o', linewidth=2.5, markersize=5, label='He + ReLU ✓')
ax.plot(layer_idx, variances_xavier_relu, 'b--s', linewidth=2, markersize=4,
        label='Xavier + ReLU ✗', alpha=0.8)
ax.axhline(y=0.5, color='orange', linestyle=':', linewidth=2,
           label='ReLU 후 목표 ≈ 0.5', alpha=0.8)
ax.set_title('일반 스케일 비교\n→ Xavier+ReLU는 빠르게 소실됨', fontsize=11)
ax.set_xlabel('레이어 번호')
ax.set_ylabel('분산')
ax.set_ylim(-0.1, 3.0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 오른쪽: 로그 스케일 (소실 정도 명확히 보기)
ax = axes[1]
ax.semilogy(layer_idx, variances_he, 'g-o', linewidth=2.5, markersize=5, label='He + ReLU ✓')
ax.semilogy(layer_idx, [max(v, 1e-300) for v in variances_xavier_relu],
            'b--s', linewidth=2, markersize=4, label='Xavier + ReLU ✗', alpha=0.8)
ax.axhline(y=0.5, color='orange', linestyle=':', linewidth=2,
           label='목표 ≈ 0.5', alpha=0.8)
ax.set_title('로그 스케일 비교\n→ Xavier+ReLU 소실 정도 명확히 보임', fontsize=11)
ax.set_xlabel('레이어 번호')
ax.set_ylabel('분산 (log scale)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("→ He 초기화(초록)는 ReLU 이후에도 분산이 안정적으로 유지됩니다.")
print("→ Xavier(파랑)는 ReLU와 함께 사용하면 분산이 빠르게 소실됩니다.")
print()
print("실무 팁:")
print("  활성화 함수가 ReLU / LeakyReLU / ELU  → He 초기화")
print("  활성화 함수가 tanh / sigmoid           → Xavier 초기화")


---

## 섹션 6: GPT-2 초기화 — Transformer 잔차 네트워크용

### Transformer의 특별한 구조: 잔차 연결 (Residual Connection)

GPT-2, BERT, LLaMA 등 Transformer 계열 모델에는 **잔차 연결**이라는 구조가 있습니다:

```python
# 일반 레이어:
x = Layer(x)           # x를 레이어에 통과시켜 새 x로 교체

# 잔차 연결:
x = x + Layer(x)      # 레이어 출력을 기존 x에 더함!
                       # → "잔차(residual)" = Layer(x)가 x에 더해지는 차이값
```

잔차 연결의 장점: 그래디언트가 레이어를 건너뛰어 직접 흐를 수 있어 학습이 안정됨.

### 잔차 연결이 만드는 분산 문제

```
레이어를 L번 쌓으면:
  x₁ = x₀ + F₁(x₀)   →   Var(x₁) ≈ Var(x₀) + Var(F₁(x₀))
  x₂ = x₁ + F₂(x₁)   →   Var(x₂) ≈ Var(x₀) + Var(F₁) + Var(F₂)
  ...
  xₗ              →   Var(xₗ) ≈ Var(x₀) + L × Var(각 잔차)
                                           ↑
                                          L배 누적 증가!
```

### GPT-2의 해결책

잔차에 기여하는 가중치의 std를 더 작게 초기화합니다:

```
일반 가중치 (Q, K, V, Embedding 등):
  std = 0.02

잔차 연결 가중치 (Attention output, FFN output):
  std = 0.02 / √(2 × n_layers)
       ↑       ↑
     기본값   각 레이어당 잔차 2번 (Attention + FFN)

→ L개 레이어 후 누적 분산: L × (0.02/√(2L))² = (0.02)²/2 ≈ 일정!
```


In [ ]:
# ─────────────────────────────────────────────────────────────────
# GPT-2 스타일 초기화를 시뮬레이션합니다.
#
# 각 Transformer 레이어는:
#   1) Attention 잔차 연결: x = x + Attention_output(x)
#   2) FFN 잔차 연결:       x = x + FFN_output(x)
#
# 잔차 연결 가중치에 std_residual을 쓰면 누적 분산이 안정됩니다.
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("[섹션 6] GPT-2 초기화 — Transformer 잔차 연결")
print("=" * 60)

n_layers_gpt = 12    # GPT-2 base: 12개 레이어
d_model = 768         # GPT-2 base: 768 차원

std_normal   = 0.02
std_residual = 0.02 / np.sqrt(2 * n_layers_gpt)   # 잔차 연결용

print()
print("📐 GPT-2 초기화 설정 (GPT-2 Base 기준):")
print(f"  d_model       = {d_model}")
print(f"  n_layers      = {n_layers_gpt}")
print(f"  잔차 연결 수  = 레이어당 2회 (Attention + FFN)")
print()
print(f"  일반 가중치 std   = {std_normal}")
print(f"  잔차 가중치 std   = 0.02 / sqrt(2 × {n_layers_gpt})")
print(f"                    = 0.02 / {np.sqrt(2 * n_layers_gpt):.4f}")
print(f"                    = {std_residual:.6f}")

# Transformer 입력: (배치, 시퀀스 길이, 모델 차원)
batch   = 4
seq_len = 32

np.random.seed(42)
x_gpt2  = np.random.randn(batch, seq_len, d_model) * std_normal
x_naive = x_gpt2.copy()   # 같은 시작점에서 비교

print()
print(f"입력 shape: ({batch}, {seq_len}, {d_model})")
print(f"           = (배치, 시퀀스 길이, 모델 차원)")
print(f"초기 분산: {np.var(x_gpt2):.6f}")
print()
print("─" * 70)
print(f"  {'레이어':^6} | {'GPT-2 초기화':^22} | {'단순 초기화(0.02)':^22}")
print("  " + "─" * 58)

variances_gpt2  = [np.var(x_gpt2)]
variances_naive = [np.var(x_naive)]

for l in range(n_layers_gpt):
    seed_l = l * 7 + 3   # 레이어마다 다른 시드 (단, 두 방법은 동일 시드로 공정 비교)

    # ─── GPT-2 방식: 잔차 가중치에 std_residual 사용 ─────────────
    np.random.seed(seed_l)
    W_attn  = np.random.randn(d_model, d_model) * std_residual
    attn_out = x_gpt2 @ W_attn
    x_gpt2 = x_gpt2 + attn_out          # 잔차 연결 (더하기)

    W_ffn   = np.random.randn(d_model, d_model) * std_residual
    ffn_out  = x_gpt2 @ W_ffn
    x_gpt2 = x_gpt2 + ffn_out           # 잔차 연결 (더하기)

    # ─── 단순 방식: 모든 가중치에 std_normal(0.02) 사용 ──────────
    np.random.seed(seed_l)
    W_attn_n  = np.random.randn(d_model, d_model) * std_normal
    attn_out_n = x_naive @ W_attn_n
    x_naive = x_naive + attn_out_n

    W_ffn_n   = np.random.randn(d_model, d_model) * std_normal
    ffn_out_n  = x_naive @ W_ffn_n
    x_naive = x_naive + ffn_out_n

    v_gpt2  = np.var(x_gpt2)
    v_naive = np.var(x_naive)
    variances_gpt2.append(v_gpt2)
    variances_naive.append(v_naive)

    print(f"  {l+1:^6} | {v_gpt2:^22.6f} | {v_naive:^22.6f}")

print()
print("─" * 60)
print(f"최종 분산 비교:")
print(f"  GPT-2 초기화: {variances_gpt2[-1]:.6f}")
print(f"  단순 초기화:  {variances_naive[-1]:.6f}")
drift = variances_naive[-1] / variances_gpt2[-1] if variances_gpt2[-1] > 0 else float('inf')
print(f"  차이 배율:    {drift:.1f}배")
print()
print("💡 GPT-2 초기화는 잔차 연결의 분산 누적을 효과적으로 제어합니다!")


In [ ]:
# GPT-2 초기화 결과 시각화

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('GPT-2 Initialization: Residual Variance Control\n(GPT-2 초기화: 잔차 연결 분산 제어)',
             fontsize=13, fontweight='bold', y=1.02)

layer_idx_gpt = list(range(n_layers_gpt + 1))

# 왼쪽: 절대적 분산 변화
ax = axes[0]
ax.plot(layer_idx_gpt, variances_gpt2,  'g-o', linewidth=2.5, markersize=8,
        label='GPT-2 초기화 ✓')
ax.plot(layer_idx_gpt, variances_naive, 'r--s', linewidth=2, markersize=7,
        label='단순 초기화 ✗', alpha=0.8)
ax.set_title('레이어별 분산 절대값\n→ GPT-2만 안정적으로 유지', fontsize=11)
ax.set_xlabel('레이어 번호 (Transformer Block)')
ax.set_ylabel('분산 (Variance)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 오른쪽: 초기 분산 대비 상대 변화
ax = axes[1]
v0_g = variances_gpt2[0]
v0_n = variances_naive[0]
ax.plot(layer_idx_gpt, [v / v0_g for v in variances_gpt2], 'g-o', linewidth=2.5, markersize=8,
        label='GPT-2 초기화 ✓')
ax.plot(layer_idx_gpt, [v / v0_n for v in variances_naive], 'r--s', linewidth=2, markersize=7,
        label='단순 초기화 ✗', alpha=0.8)
ax.axhline(y=1.0, color='black', linestyle=':', linewidth=2, alpha=0.5, label='기준=1.0')
ax.set_title('초기 분산 대비 상대 변화\n→ 단순 초기화는 레이어마다 누적 증가', fontsize=11)
ax.set_xlabel('레이어 번호')
ax.set_ylabel('분산 / 초기 분산 (상대값)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("수치로 확인:")
print(f"  잔차 연결 횟수 = {n_layers_gpt} 레이어 × 2 = {2*n_layers_gpt}회")
print(f"  단순 초기화: 각 잔차의 분산 기여 = {std_normal}² = {std_normal**2:.6f}")
print(f"  단순 초기화: 총 누적 기여 = {2*n_layers_gpt} × {std_normal**2:.6f} = {2*n_layers_gpt * std_normal**2:.4f}")
print()
print(f"  GPT-2 초기화: 각 잔차의 분산 기여 = {std_residual:.6f}² = {std_residual**2:.9f}")
print(f"  GPT-2 초기화: 총 누적 기여 = {2*n_layers_gpt} × {std_residual**2:.9f} = {2*n_layers_gpt * std_residual**2:.6f}")
print()
print("→ GPT-2는 누적 분산 기여를 훨씬 작게 유지합니다!")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 섹션 7: 종합 비교
#
# 지금까지 배운 모든 초기화 방법을 동일한 조건에서 비교합니다.
#
# run_experiment(): 초기화 함수와 활성화 함수를 받아
#                   50개 레이어 통과 후의 분산 변화를 반환
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("[섹션 7] 종합 비교 — 모든 초기화 방법")
print("=" * 60)

d = 512
n_layers = 50

def run_experiment(init_fn, act_fn=None, seed=42):
    # 초기화 함수와 활성화 함수를 받아 각 레이어 후 분산을 기록
    np.random.seed(seed)
    x = np.random.randn(d)
    variances = [np.var(x)]
    for _ in range(n_layers):
        W = init_fn(d)
        x = W @ x
        if act_fn is not None:
            x = act_fn(x)
        variances.append(np.var(x))
    return variances

# 각 방법별 초기화 함수 정의 (lambda: 한 줄 함수)
experiments = {
    '나쁜: std=1.0 (폭발)':  (lambda d: np.random.randn(d, d) * 1.0,                       None),
    '나쁜: std=0.01 (소실)': (lambda d: np.random.randn(d, d) * 0.01,                      None),
    'Xavier (선형/tanh)':    (lambda d: np.random.randn(d, d) * np.sqrt(2.0 / (d + d)),   None),
    'He + ReLU':             (lambda d: np.random.randn(d, d) * np.sqrt(2.0 / d),          relu),
    'Xavier + ReLU (비교)':  (lambda d: np.random.randn(d, d) * np.sqrt(2.0 / (d + d)),   relu),
}

print()
print(f"실험 설정: d={d} 차원, {n_layers}개 레이어")
print()
print(f"  {'방법':<28} | {'초기 분산':>10} | {'최종 분산':>12} | 판정")
print("  " + "─" * 70)

all_variances = {}
for name, (init_fn, act_fn) in experiments.items():
    variances = run_experiment(init_fn, act_fn)
    all_variances[name] = variances
    init_v  = variances[0]
    final_v = variances[-1]

    if np.isnan(final_v) or np.isinf(final_v) or final_v > 1e10:
        verdict = "폭발 ❌"
    elif final_v < 1e-10:
        verdict = "소실 ❌"
    elif 0.1 < final_v < 10.0:
        verdict = "안정 ✅"
    else:
        verdict = "불안정 ⚠️"

    print(f"  {name:<28} | {init_v:>10.4f} | {final_v:>12.4e} | {verdict}")

print()
print("─" * 60)
print()
print("💡 Xavier와 He 모두 분산을 안정적으로 유지합니다.")
print("   Xavier + ReLU는 ReLU와 맞지 않아 소실 발생에 주의하세요.")


In [ ]:
# 모든 방법을 2×2 그리드로 시각화합니다.

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('종합 비교: 가중치 초기화 방법별 분산 안정성\n(Comprehensive Comparison of Weight Initialization)',
             fontsize=14, fontweight='bold')

layer_idx = list(range(n_layers + 1))

COLORS = {
    '나쁜: std=1.0 (폭발)':  ('crimson',      '--'),
    '나쁜: std=0.01 (소실)': ('royalblue',    '--'),
    'Xavier (선형/tanh)':    ('forestgreen',  '-'),
    'He + ReLU':             ('darkorange',   '-'),
    'Xavier + ReLU (비교)':  ('mediumpurple', ':'),
}

# 왼쪽 위: 전체 비교 (로그 스케일)
ax = axes[0, 0]
for name, variances in all_variances.items():
    color, ls = COLORS[name]
    safe = [min(max(v, 1e-300), 1e50) if not (np.isnan(v) or np.isinf(v)) else 1e50
            for v in variances]
    ax.semilogy(layer_idx, safe, color=color, linestyle=ls, linewidth=2, label=name)
ax.axhline(1.0, color='black', linestyle=':', linewidth=1.5, alpha=0.5, label='목표=1.0')
ax.set_title('전체 비교 (로그 스케일)', fontsize=11)
ax.set_xlabel('레이어 번호')
ax.set_ylabel('분산 (log scale)')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)

# 오른쪽 위: 안정 방법 확대 (일반 스케일)
ax = axes[0, 1]
for name in ['Xavier (선형/tanh)', 'He + ReLU']:
    color, ls = COLORS[name]
    ax.plot(layer_idx, all_variances[name], color=color, linestyle=ls,
            linewidth=2.5, marker='o', markersize=4, label=name)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='기준=1.0')
ax.fill_between(layer_idx, 0.1, 5.0, alpha=0.07, color='green', label='허용 범위')
ax.set_title('안정적인 방법 확대 보기', fontsize=11)
ax.set_xlabel('레이어 번호')
ax.set_ylabel('분산')
ax.set_ylim(0, 4)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 왼쪽 아래: 나쁜 초기화
ax = axes[1, 0]
for name in ['나쁜: std=1.0 (폭발)', '나쁜: std=0.01 (소실)']:
    color, ls = COLORS[name]
    safe = [min(max(v, 1e-300), 1e50) if not (np.isnan(v) or np.isinf(v)) else 1e50
            for v in all_variances[name]]
    ax.semilogy(layer_idx, safe, color=color, linestyle=ls, linewidth=2.5, label=name)
ax.axhline(1.0, color='green', linestyle='--', linewidth=2, alpha=0.8, label='목표=1.0')
ax.set_title('나쁜 초기화 상세', fontsize=11)
ax.set_xlabel('레이어 번호')
ax.set_ylabel('분산 (log scale)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#fff5f5')

# 오른쪽 아래: 요약 텍스트 표
ax = axes[1, 1]
ax.axis('off')
table_data = [
    ['방법', '공식', '활성화', '안정성'],
    ['0 초기화',       'W = 0',              '-',            '불가 ❌'],
    ['std=1.0',        'W~N(0,1)',            '-',            '폭발 ❌'],
    ['std=0.01',       'W~N(0,0.01)',         '-',            '소실 ❌'],
    ['Xavier',         '√(2/(fin+fout))',     'tanh/sig',     '안정 ✅'],
    ['He',             '√(2/fan_in)',         'ReLU',         '안정 ✅'],
    ['GPT-2 Residual', '0.02/√(2L)',          'Transformer',  '안정 ✅'],
]
tbl = ax.table(
    cellText=table_data[1:], colLabels=table_data[0],
    cellLoc='center', loc='center',
    bbox=[0.0, 0.1, 1.0, 0.85]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif '✅' in cell.get_text().get_text():
        cell.set_facecolor('#d5f5e3')
    elif '❌' in cell.get_text().get_text():
        cell.set_facecolor('#fadbd8')
    cell.set_edgecolor('#bdc3c7')
ax.set_title('초기화 방법 요약', fontsize=12, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()
print()
print("→ 오른쪽 위 그래프: Xavier와 He 모두 분산을 안정적으로 유지합니다.")
print("→ 왼쪽 아래 그래프: 스케일 없는 초기화의 폭발/소실이 명확합니다.")


---

## 실제 PyTorch에서의 사용법

실제 프로젝트에서는 NumPy 대신 PyTorch의 내장 초기화 함수를 사용합니다.

```python
import torch
import torch.nn as nn

layer = nn.Linear(512, 512)

# ─── Xavier 초기화 ───────────────────────────────────────
nn.init.xavier_uniform_(layer.weight)    # 균등 분포 버전
nn.init.xavier_normal_(layer.weight)     # 정규 분포 버전 (우리가 구현한 것)

# ─── He 초기화 (Kaiming) ─────────────────────────────────
nn.init.kaiming_uniform_(layer.weight, mode='fan_in', nonlinearity='relu')
nn.init.kaiming_normal_(layer.weight,  mode='fan_in', nonlinearity='relu')

# ─── GPT-2 스타일 ────────────────────────────────────────
n_layers = 12
# 일반 가중치
nn.init.normal_(layer.weight, mean=0.0, std=0.02)

# 잔차 연결 가중치 (더 작게!)
nn.init.normal_(layer.weight, mean=0.0, std=0.02 / (2 * n_layers) ** 0.5)
```

> 💡 **알아두세요**: PyTorch의 `nn.Linear`, `nn.Conv2d` 등은  
> **기본값으로 이미 Kaiming Uniform(He 초기화)** 을 사용합니다!  
> 대부분의 경우 별도 설정 없이도 OK입니다.

---

### 언제 어떤 초기화를?

| 모델 유형 | 활성화 함수 | 권장 초기화 |
|-----------|------------|------------|
| CNN, MLP | ReLU, LeakyReLU, ELU | He (Kaiming) |
| CNN, MLP | tanh, sigmoid | Xavier (Glorot) |
| Transformer, GPT | GELU, ReLU | GPT-2 스타일 |
| 기타 커스텀 | 실험적 | 분산 추적 후 선택 |


---

## 📌 핵심 정리

### 왜 가중치 초기화가 중요한가?

레이어를 여러 번 통과할수록 **분산이 폭발하거나 소실**될 수 있습니다.  
올바른 초기화는 레이어를 많이 쌓아도 **분산이 ≈ 1로 안정적으로 유지**되게 합니다.

---

### 각 초기화 방법 한 줄 요약

| 방법 | 공식 | 한 줄 요약 |
|------|------|-----------|
| **0 초기화** | `W = 0` | ❌ 대칭성 파괴 실패 → 절대 사용 금지 |
| **스케일 없는 랜덤** | `W ~ N(0,1)` | ❌ 분산 폭발/소실 → 사용 금지 |
| **Xavier** | `std = √(2/(fan_in+fan_out))` | ✅ tanh/sigmoid에 최적 |
| **He (Kaiming)** | `std = √(2/fan_in)` | ✅ ReLU에 최적 |
| **GPT-2 잔차** | `std = 0.02/√(2×L)` | ✅ Transformer 잔차 연결에 최적 |

---

### 실무에서 기억할 3가지

1. **ReLU 계열 → He (Kaiming)** `nn.init.kaiming_normal_()`
2. **tanh/sigmoid 계열 → Xavier** `nn.init.xavier_normal_()`
3. **Transformer → GPT-2 스타일** 잔차 가중치에 `1/√(2L)` 스케일 적용

---

### 🔬 더 탐구해볼 것

- **배치 정규화 (Batch Normalization)**: 초기화를 덜 민감하게 만드는 기법
- **레이어 정규화 (Layer Normalization)**: Transformer에서 사용하는 정규화
- **Spectral Normalization**: GAN 훈련 안정화
- **LoRA**: 파라미터 효율적 파인튜닝에서의 초기화 전략
